# core

> Serve Python functions as MCP tools, and call MCP servers

In [ ]:
#| default_exp core

Serve Python functions as MCP tools, or call another MCP server from Python.

## Serving tools

Pass sync or async functions to `MCPServer`. Fastcore's `get_schema` reads their docments to produce tool descriptions and `inputSchema`. No separate registration code is needed.

The server dispatches message dictionaries without keeping a client session. Its stdio transport handles concurrent calls and nested requests. A tool can await `MCPServer.elicit` to ask for structured user input while its call remains active. For HTTP, `create_app` supplies a mountable ASGI app with one POST endpoint; `serve_mcp` runs it under uvicorn.

## Auth

`auth_app` checks a static bearer token with a constant-time comparison. It is ASGI middleware, not an OAuth implementation. Treat access to your tools as permission to run their Python code. `serve_mcp` refuses a non-loopback bind without `token=` or `$MCPMINI_TOKEN`. Use `no_token=True` only when you provide protection elsewhere, such as a VPN or authenticating proxy.

## The command

`mcpmini tools.py` serves a file's docmented functions over stdio. This is the command an MCP host can launch. Add `--transport http` and the deployment flags for HTTP. From Python, `load_server` performs the same file-to-server conversion. The file's docstring supplies the server instructions.

## Calling servers

Use `MCPClient.stdio(argv)` or `MCPClient.http(url)` as an async context manager. The client initializes the connection and binds `tools/list` entries as Python callables. Fastcore's `mk_tool` reconstructs their signatures, documentation, and defaults from the schemas.

Bound tools return text and raise on `isError`. Use `call_tool` for the raw result, including non-text blocks. A stdio client's `on_request(method, params)` can answer nested requests such as elicitation. The HTTP client accepts JSON or SSE replies and returns a server-issued `Mcp-Session-Id` on later POSTs. Its `delete` method requests session termination when you no longer need a server-side session.

The tests below pair each side with the official SDK. These cover the supported tool workflow, not every MCP feature or server.


This implementation negotiates the `2025-11-25`, `2025-06-18`, and `2025-03-26` protocol versions. They use an `initialize` handshake before ordinary JSON-RPC calls.

The [2026-07-28 revision](https://blog.modelcontextprotocol.io/posts/2026-07-28/) removes that handshake and session header in favor of self-describing requests. Keeping connection handling outside `dispatch` makes that change easier to accommodate, but this code does not yet implement the new revision.

In [ ]:
#| export
import asyncio, contextlib, contextvars, httpx, importlib.util, inspect, json, secrets, socket, sys, traceback
from fastcore.utils import *
from fastcore.meta import delegates
from fastcore.script import call_parse
from fastcore.funccall import get_schema, mk_tool
from starlette.applications import Starlette
from starlette.responses import JSONResponse, Response
from starlette.routing import Route
from mcpmini import __version__


In [ ]:
from fastcore.test import *
import shutil, socket, tempfile

## Messages

`jreq` builds a JSON-RPC request when you pass an id, or a notification without one. `jtool` builds a `tools/call` request and supplies an empty `arguments` object when no arguments are given.


In [ ]:
#| export
def jreq(method, id=None, **params):
    "JSON-RPC request, or notification when `id` is None."
    res = dict(jsonrpc='2.0', method=method)
    if id is not None: res['id'] = id
    if params: res['params'] = params
    return res

def jtool(name, id=1, **arguments):
    "A `tools/call` request; omitted arguments become an empty object."
    return jreq('tools/call', id, name=name, arguments=arguments)

Use `jresp` for a result and `jerr` for a JSON-RPC error reply.

In [ ]:
#| export
def jresp(id, result): return dict(jsonrpc='2.0', id=id, result=result)
def jerr(id, code, message):
    "JSON-RPC error reply; MCP uses -32601 for unknown methods, -32602 for bad params, -32603 for handler failures"
    return dict(jsonrpc='2.0', id=id, error=dict(code=code, message=message))

Stdio carries one JSON message per line. `send_jmsg` writes that line; `recv_jmsg` reads it and returns `None` at EOF.

In [ ]:
#| export
async def send_jmsg(writer, msg):
    "Write one newline-delimited JSON message."
    writer.write(json.dumps(msg).encode()+b'\n')
    await writer.drain()

async def recv_jmsg(reader):
    "Read one newline-delimited JSON message; None at EOF."
    if line := await reader.readline(): return json.loads(line)

## Tools from functions

A tool is just a docmented function. Its name, docstring, annotations, and parameter comments give `get_schema` the information MCP needs. Pass `pname='inputSchema'` to use MCP's name for the argument schema:

In [ ]:
def fahrenheit(
    celsius:float, # Temperature to convert
)->float:
    "Convert Celsius to Fahrenheit"
    return celsius*9/5+32

get_schema(fahrenheit, pname='inputSchema')

{'name': 'fahrenheit',
 'description': 'Convert Celsius to Fahrenheit\n\nReturns:\n- type: number',
 'inputSchema': {'type': 'object',
  'properties': {'celsius': {'description': 'Temperature to convert',
    'type': 'number'}},
  'required': ['celsius']}}

MCP tool results contain a list of content blocks. `_content` puts a string in one text block and uses `repr` for other values. This keeps Python's representation, including quotes around nested strings. A dict with a `content` key passes through unchanged, allowing tools to return images or several blocks.

In [ ]:
#| export
def _content(res):
    "A `tools/call` result for return value `res`"
    if isinstance(res, dict) and 'content' in res: return res
    return dict(content=[dict(type='text', text=res if isinstance(res, str) else repr(res))], isError=False)

In [ ]:
test_eq(_content('hi')['content'], [dict(type='text', text='hi')])
test_eq(_content(212.0)['content'][0]['text'], '212.0')
passthru = dict(content=[dict(type='image', data='AAAA', mimeType='image/png')])
test_eq(_content(passthru), passthru)
_content(dict(a=1))

{'content': [{'type': 'text', 'text': "{'a': 1}"}], 'isError': False}

## The server

`MCPServer` stores its name, version, optional model instructions, and the tools' functions and schemas. Transports and tests both call `dispatch` with a message dict. The server keeps no client-session table. A context variable holds the transport callback for the current tool call.

In [ ]:
#| export
PROTO_VERSIONS = '2025-11-25','2025-06-18','2025-03-26'

class MCPServer:
    "An MCP tool server: `dispatch` maps one message dict to one reply dict"
    def __init__(self,
        name, # Server name reported to clients
        tools=(), # Functions (sync or async) exposed as tools, described by their docments
        version=__version__, # Server version reported to clients
        instructions=None, # Optional guidance forwarded to the client's model
    ):
        self.name,self.version,self.instructions = name,version,instructions
        self.specs = [get_schema(f, pname='inputSchema') for f in tools]
        self.funcs = {s['name']:f for s,f in zip(self.specs,tools)}
        self._requester = contextvars.ContextVar(f'{name}_requester', default=None)
    async def request(self, method, **params):
        "Send a request through the transport serving the current tool call."
        if (requester := self._requester.get()) is None: raise RuntimeError('this tool call has no bidirectional transport')
        return await requester(method, params)
    async def elicit(self, message, schema):
        "Request structured user input during the current tool call."
        return await self.request('elicitation/create', message=message, requestedSchema=schema)

`initialize` echoes a supported version or offers the newest entry in `PROTO_VERSIONS`. Unknown tool names receive JSON-RPC errors. Exceptions raised by tools instead return a traceback in `content` with `isError=True`, where the model can read and respond to the failure.

In [ ]:
#| export
async def _acall(f, args):
    "A `tools/call` result from calling `f`, exceptions in-band as `isError` content"
    try:
        res = f(**args)
        if inspect.iscoroutine(res): res = await res
        return _content(res)
    except Exception: return dict(content=[dict(type='text', text=traceback.format_exc())], isError=True)

@patch
async def dispatch(self:MCPServer, msg, requester=None):
    "The reply dict for one JSON-RPC message dict; None for notifications"
    id,meth,p = msg.get('id'),msg.get('method',''),msg.get('params',{})
    if 'id' not in msg: return None
    try:
        if meth=='initialize':
            pv = p.get('protocolVersion')
            r = dict(protocolVersion=pv if pv in PROTO_VERSIONS else PROTO_VERSIONS[0],
                capabilities=dict(tools={}), serverInfo=dict(name=self.name, version=self.version))
            if self.instructions: r['instructions'] = self.instructions
            return jresp(id, r)
        if meth=='tools/list': return jresp(id, dict(tools=self.specs))
        if meth=='tools/call':
            f = self.funcs.get(p['name'])
            if not f: return jerr(id, -32602, f"Unknown tool: {p['name']}")
            token = self._requester.set(requester)
            try: return jresp(id, await _acall(f, p.get('arguments',{})))
            finally: self._requester.reset(token)
        if meth=='ping': return jresp(id, {})
        return jerr(id, -32601, f'Method not found: {meth}')
    except Exception as e: return jerr(id, -32603, f'{type(e).__name__}: {e}')

Our demo server converts temperatures, takes an async nap, and crashes on demand. The initialization tests request one supported version and one unsupported version:

In [ ]:
async def nap(
    ms:int=1, # How long to sleep, in milliseconds
)->str:
    "Sleep, then report back"
    await asyncio.sleep(ms/1000)
    return f'napped {ms}ms'

def crash()->str:
    "Always fails"
    return 1/0

srv = MCPServer('demo', [fahrenheit, nap, crash], instructions='Convert temperatures on request.')
init = await srv.dispatch(jreq('initialize', 1, protocolVersion='2025-06-18', capabilities={}, clientInfo=dict(name='t', version='0')))
test_eq(init['result']['protocolVersion'], '2025-06-18')
old = await srv.dispatch(jreq('initialize', 2, protocolVersion='2024-11-05'))
test_eq(old['result']['protocolVersion'], PROTO_VERSIONS[0])
init['result']

{'protocolVersion': '2025-06-18',
 'capabilities': {'tools': {}},
 'serverInfo': {'name': 'demo', 'version': '0.0.5'},
 'instructions': 'Convert temperatures on request.'}

Both synchronous functions and coroutines return MCP content blocks through `tools/call`:

In [ ]:
fahrenheit_reply = await srv.dispatch(jtool('fahrenheit', celsius=100))
nap_reply = await srv.dispatch(jtool('nap', ms=2))
texts = [r['result']['content'][0]['text'] for r in (fahrenheit_reply, nap_reply)]
test_eq(texts, ['212.0','napped 2ms'])
texts

['212.0', 'napped 2ms']

Calling `crash` returns its traceback as a tool result, not a JSON-RPC error:

In [ ]:
boom = (await srv.dispatch(jtool('crash')))['result']
assert boom['isError'] and 'ZeroDivisionError' in boom['content'][0]['text']
boom

{'content': [{'type': 'text',
   'text': 'Traceback (most recent call last):\n  File "<ipython-input-11-171bf1eed883>", line 5, in _acall\n    res = f(**args)\n  File "<ipython-input-12-2bcae28d6802>", line 10, in crash\n    return 1/0\n           ~^~\nZeroDivisionError: division by zero\n'}],
 'isError': True}

## stdio

`serve_stdio` reads newline-delimited JSON until EOF. A malformed JSON line receives error `-32700` with a null id, since no request id is available. Notifications receive no reply. You can supply asyncio streams or use the default stdin and stdout.

For tests, `stdio_peer` connects the server and a small `StdioPeer` client through a socketpair. The helper handles setup and shutdown; the messages still use the real wire format.

In [ ]:
#| export
class StdioPeer:
    "A small JSON-RPC peer connected to `serve_stdio`, for examples and protocol tests."
    def __init__(self, reader, writer): self.reader,self.writer,self.id = reader,writer,0
    async def send(self, msg): await send_jmsg(self.writer, msg)
    async def send_raw(self, line):
        self.writer.write(line)
        await self.writer.drain()
    async def recv(self): return await recv_jmsg(self.reader)
    async def send_request(self, method, **params):
        self.id += 1
        await self.send(jreq(method, self.id, **params))
        return self.id
    async def request(self, method, **params):
        await self.send_request(method, **params)
        return await self.recv()
    async def send_call(self, name, **arguments): return await self.send_request('tools/call', name=name, arguments=arguments)
    async def call(self, name, **arguments):
        await self.send_call(name, **arguments)
        return await self.recv()
    async def reply(self, request, result): await self.send(jresp(request['id'], result))

@contextlib.asynccontextmanager
async def stdio_peer(srv):
    "An in-process `StdioPeer` speaking the real newline-delimited wire protocol to `srv`."
    server_sock,client_sock = socket.socketpair()
    server_reader,server_writer = await asyncio.open_connection(sock=server_sock)
    client_reader,client_writer = await asyncio.open_connection(sock=client_sock)
    task = asyncio.create_task(serve_stdio(srv, server_reader, server_writer))
    try: yield StdioPeer(client_reader, client_writer)
    finally:
        client_writer.close()
        await client_writer.wait_closed()
        try: await task
        finally:
            server_writer.close()
            await server_writer.wait_closed()


In [ ]:
#| export
async def _stdio_streams():
    loop = asyncio.get_running_loop()
    reader = asyncio.StreamReader()
    await loop.connect_read_pipe(lambda: asyncio.StreamReaderProtocol(reader), sys.stdin)
    tr,pr = await loop.connect_write_pipe(asyncio.streams.FlowControlMixin, sys.stdout)
    return reader,asyncio.StreamWriter(tr, pr, None, loop)

async def serve_stdio(
    srv, # `MCPServer` to serve
    reader=None, # asyncio stream to read messages from; stdin if None
    writer=None, # asyncio stream to write replies to; stdout if None
):
    "Serve `srv` over newline-delimited JSON-RPC until EOF, dispatching messages concurrently"
    if reader is None: reader,writer = await _stdio_streams()
    tasks,pending,wlock = set(),{},asyncio.Lock()
    request_id = 0
    async def _write(msg):
        async with wlock: await send_jmsg(writer, msg)
    async def _request(method, params):
        nonlocal request_id
        request_id += 1
        id = f'server-{request_id}'
        fut = pending[id] = asyncio.get_running_loop().create_future()
        await _write(jreq(method, id, **params))
        try:
            resp = await fut
            if 'error' in resp: raise RuntimeError(f"{resp['error']['code']}: {resp['error']['message']}")
            return resp['result']
        finally: pending.pop(id, None)
    async def _one(msg):
        resp = jerr(None, -32700, 'Parse error') if msg is None else await srv.dispatch(msg, _request)
        if resp is not None: await _write(resp)
    while line := await reader.readline():
        if not line.strip(): continue
        try: msg = json.loads(line)
        except Exception: msg = None
        if msg is not None and 'method' not in msg and (fut := pending.get(msg.get('id'))) is not None:
            if not fut.done(): fut.set_result(msg)
            continue
        task = asyncio.create_task(_one(msg))
        tasks.add(task)
        task.add_done_callback(tasks.discard)
    for fut in pending.values():
        if not fut.done(): fut.set_exception(EOFError('MCP client closed'))
    if tasks: await asyncio.gather(*tasks, return_exceptions=True)

The peer calls a tool, sends malformed JSON, then pings the server. The parse error does not prevent the ping from succeeding. Closing the peer ends the server task:

In [ ]:
async with stdio_peer(srv) as peer:
    r = await peer.call('fahrenheit', celsius=0)
    test_eq(r['result']['content'][0]['text'], '32.0')
    await peer.send_raw(b'not json\n')
    await peer.send_request('ping')
    test_eq((await peer.recv())['error']['code'], -32700)
    test_eq((await peer.recv())['result'], {})
r

{'jsonrpc': '2.0',
 'id': 1,
 'result': {'content': [{'type': 'text', 'text': '32.0'}], 'isError': False}}

A tool can need more information before it can finish. Await `MCPServer.elicit` to ask the client for structured input without ending the tool call. It sends `elicitation/create` through that call's transport and returns the client's response. The tool can then continue as ordinary sequential code. Here it asks for a name before returning a greeting:

In [ ]:
async def greeting()->str:
    "Ask for a name, then greet it"
    r = await interactive.elicit('Who?', dict(type='object', properties=dict(name=dict(type='string')), required=['name']))
    return f"Hello {r['content']['name']}"

interactive = MCPServer('interactive', [greeting])

The client answers the nested request before it receives the greeting. A reply has no `method`; `serve_stdio` matches its id to the pending server request:

In [ ]:
async with stdio_peer(interactive) as peer:
    await peer.send_call('greeting')
    prompt = await peer.recv()
    test_eq(prompt['method'], 'elicitation/create')
    await peer.reply(prompt, dict(action='accept', content=dict(name='Ada')))
    greeting_res = await peer.recv()
test_eq(greeting_res['result']['content'][0]['text'], 'Hello Ada')
greeting_res

{'jsonrpc': '2.0',
 'id': 1,
 'result': {'content': [{'type': 'text', 'text': 'Hello Ada'}],
  'isError': False}}

The stdio loop starts a task for each request. The short nap can finish while the long nap is awaiting sleep. This also lets an interrupt tool run alongside an async execute tool on the same connection. A synchronous tool that blocks the event loop does not yield that opportunity:

In [ ]:
async with stdio_peer(srv) as peer:
    await peer.send_call('nap', ms=300)
    await peer.send_call('nap', ms=1)
    replies = [await peer.recv() for _ in range(2)]
test_eq([r['id'] for r in replies], [2, 1])

## Streamable HTTP

`create_app` exposes one POST endpoint, `/mcp` by default. It returns JSON for requests and HTTP 202 for notifications. It creates no `Mcp-Session-Id` and ignores an incoming one. GET returns 405; there is no server-push stream.

The [2025 transport specification](https://modelcontextprotocol.io/specification/2025-11-25/basic/transports) allows a JSON reply instead of SSE and permits 405 when a server does not offer a GET stream. Our HTTP server does not send progress notifications or nested requests.

`auth_app` wraps any ASGI app. It checks the bearer token before calling that app, protecting all HTTP routes beneath it. The comparison is constant-time. This middleware handles static-token authentication, not OAuth. A deployment still needs transport security and protection against untrusted origins.

In [ ]:
#| export
def auth_app(
    app, # ASGI app to guard
    token, # The bearer token every request must carry
):
    "Wrap `app` to reject requests whose `Authorization` bearer token doesn't match `token`"
    async def _f(scope, receive, send):
        if scope['type']=='http':
            hdr = dict(scope.get('headers') or ()).get(b'authorization', b'').decode()
            if not (hdr.startswith('Bearer ') and secrets.compare_digest(hdr[7:], token)):
                await send(dict(type='http.response.start', status=401, headers=[(b'www-authenticate', b'Bearer')]))
                return await send(dict(type='http.response.body', body=b'unauthorized'))
        await app(scope, receive, send)
    return _f

def create_app(
    srv, # `MCPServer` to expose
    token=None, # Bearer token required on every request; open if None
    path='/mcp', # Endpoint path
):
    "ASGI app serving `srv` over streamable HTTP"
    async def _post(req):
        resp = await srv.dispatch(await req.json())
        return JSONResponse(resp) if resp is not None else Response(status_code=202)
    app = Starlette(routes=[Route(path, _post, methods=['POST'])])
    return auth_app(app, token) if token else app

`httpx.ASGITransport` tests the app without a socket. Missing and wrong tokens get 401 before JSON parsing. The correct token permits a tool call. Authenticated GET gets 405 and a notification gets 202:

In [ ]:
app = create_app(srv, token='sesame')
async def hpost(msg, token=None, meth='POST'):
    hdrs = {'Authorization': f'Bearer {token}'} if token else {}
    async with httpx.AsyncClient(transport=httpx.ASGITransport(app), base_url='http://t') as c:
        return await c.request(meth, '/mcp', json=msg, headers=hdrs)
ping = jreq('ping', 1)
test_eq((await hpost(ping)).status_code, 401)
test_eq((await hpost(ping, 'wrong')).status_code, 401)
test_eq((await hpost(ping)).headers['www-authenticate'], 'Bearer')
test_eq((await hpost(ping, 'sesame')).json()['result'], {})
test_eq((await hpost(ping, 'sesame', meth='GET')).status_code, 405)
test_eq((await hpost(jreq('notifications/initialized'), 'sesame')).status_code, 202)
(await hpost(jtool('fahrenheit', 2, celsius=37), 'sesame')).json()

{'jsonrpc': '2.0',
 'id': 2,
 'result': {'content': [{'type': 'text', 'text': '98.6'}], 'isError': False}}

## Serving

A tool server lets its clients run your Python functions. `serve_mcp` therefore requires a token for a non-loopback bind. Set `no_token=True` only when something else protects access, such as a VPN or fronting proxy.

This check belongs to `serve_mcp`, not `create_app`: applications that mount the latter must arrange their own protection. Jupyter's old tokenless default is the cautionary tale here.

In [ ]:
#| export
def _loopback(host): return host in ('127.0.0.1','localhost','::1')

@delegates(create_app)
async def serve_mcp(
    srv, # `MCPServer` to serve
    host='127.0.0.1', # Interface to bind
    port=8000, # Port to bind
    token=None, # Bearer token; `$MCPMINI_TOKEN` if None
    no_token=False, # Serve a non-loopback interface openly, when auth lives elsewhere (e.g. a VPN)
    **kwargs
):
    "Serve `srv` over streamable HTTP; non-loopback binds require a token unless `no_token`"
    import uvicorn
    token = token or os.environ.get('MCPMINI_TOKEN')
    if not (_loopback(host) or token or no_token): raise ValueError(f'binding {host} needs a token: pass `token`, set $MCPMINI_TOKEN, or say `no_token=True`')
    app = create_app(srv, token, **kwargs)
    await uvicorn.Server(uvicorn.Config(app, host=host, port=port, log_level='warning')).serve()

An unprotected non-loopback bind raises before opening a socket. Loopback binds may omit the token. For the client examples, however, we start a loopback server with the token `sesame`:

In [ ]:
with expect_fail(ValueError): await serve_mcp(srv, host='0.0.0.0')

def free_port():
    with socket.socket() as s:
        s.bind(('127.0.0.1', 0))
        return s.getsockname()[1]

port = free_port()
server_task = asyncio.create_task(serve_mcp(srv, port=port, token='sesame'))
url = f'http://127.0.0.1:{port}/mcp'
for _ in range(100):
    try:
        with socket.create_connection(('127.0.0.1', port), timeout=0.5): break
    except OSError: await asyncio.sleep(0.05)
url

'http://127.0.0.1:56405/mcp'

## The client

Other servers can return SSE bodies or assign session ids even though ours does neither. The HTTP client accepts both JSON and SSE. After initialization, its POSTs include the negotiated `MCP-Protocol-Version` and any server-issued `Mcp-Session-Id`.

`sse_data` parses a complete SSE body. Blank lines separate events; an event's `data:` lines combine into one JSON message.

In [ ]:
#| export
def sse_data(text):
    "JSON messages from the `data:` lines of SSE body `text`"
    res = []
    for ev in text.split('\n\n'):
        data = '\n'.join(l[5:].lstrip() for l in ev.splitlines() if l.startswith('data:'))
        if data: res.append(json.loads(data))
    return res

In [ ]:
test_eq(sse_data('event: message\ndata: {"a": 1}\n\n: keepalive\n\ndata: {"b":\ndata:  2}\n\n'), [dict(a=1), dict(b=2)])

Each transport's `send` takes a message dict and returns its reply, or `None` for a notification. `StdioTransport` starts a subprocess. While waiting for a reply, it answers server requests through the sync or async `on_request` callback. It then continues waiting for the original request id.


In [ ]:
#| export
class StdioTransport:
    "JSON-RPC to a subprocess over its stdin/stdout"
    def __init__(self,
        argv, # Server command line, e.g. `['mymcp', '--flag']`
        env=None, # Environment for the subprocess; inherited if None
        cwd=None, # Working directory for the subprocess; inherited if None
        on_request=None, # Sync or async `(method, params)` handler for a server request received during a client request
    ):
        self.argv,self.env,self.cwd,self.on_request,self.p,self.proto = argv,env,cwd,on_request,None,None
    async def start(self):
        self.p = await asyncio.create_subprocess_exec(*self.argv, stdin=asyncio.subprocess.PIPE, stdout=asyncio.subprocess.PIPE,
            env=self.env, cwd=self.cwd)
    async def _answer(self, msg):
        try:
            if self.on_request is None: raise RuntimeError(f"no handler for server request {msg['method']}")
            result = self.on_request(msg['method'], msg.get('params',{}))
            if inspect.isawaitable(result): result = await result
            return jresp(msg['id'], result)
        except Exception as e: return jerr(msg['id'], -32603, f'{type(e).__name__}: {e}')
    async def send(self, msg):
        await send_jmsg(self.p.stdin, msg)
        if 'id' not in msg: return None
        while resp := await recv_jmsg(self.p.stdout):
            if 'method' in resp:
                if 'id' in resp: await send_jmsg(self.p.stdin, await self._answer(resp))
                continue
            if resp.get('id')==msg['id']: return resp
        raise RuntimeError(f'server exited awaiting reply {msg["id"]}')
    async def aclose(self):
        if not self.p: return
        self.p.stdin.close()
        await self.p.wait()

`HTTPTransport` sends one POST per message. For SSE, it parses the response body and selects the message with the matching id.

Call `delete` when you no longer need the server-side session. It returns the HTTP response, or `None` when no session id is set. Success and 404 clear the id. A 405 response leaves it in place because the server refused termination. Other HTTP errors raise. `aclose` closes the HTTP client without requesting session termination.

In [ ]:
#| export
class HTTPTransport:
    "JSON-RPC to a streamable HTTP endpoint, one POST per message"
    def __init__(self,
        url, # The MCP endpoint, e.g. 'http://127.0.0.1:8000/mcp'
        token=None, # Bearer token to send with every request
        http_client=None, # An `httpx.AsyncClient` to use, e.g. over `httpx.ASGITransport`; a fresh one if None
    ):
        self.headers = {'Accept': 'application/json, text/event-stream'}
        if token: self.headers['Authorization'] = f'Bearer {token}'
        self.url,self.client,self.sess,self.proto = url,http_client,None,None
    async def start(self):
        if self.client is None: self.client = httpx.AsyncClient()
    def _request_headers(self):
        h = dict(self.headers)
        if self.sess: h['Mcp-Session-Id'] = self.sess
        if self.proto: h['MCP-Protocol-Version'] = self.proto
        return h
    async def send(self, msg):
        r = await self.client.post(self.url, json=msg, headers=self._request_headers())
        r.raise_for_status()
        if sid := r.headers.get('mcp-session-id'): self.sess = sid
        if 'id' not in msg: return None
        if r.headers.get('content-type','').startswith('text/event-stream'):
            return first(m for m in sse_data(r.text) if m.get('id')==msg['id'])
        return r.json()
    async def delete(self):
        "Request session termination and return the HTTP response, or None without a session."
        if not self.sess: return
        r = await self.client.delete(self.url, headers=self._request_headers())
        if r.status_code == 405: return r
        if r.status_code != 404: r.raise_for_status()
        self.sess = None
        return r
    async def aclose(self): await self.client.aclose()

`MCPClient.start` initializes the transport, sends `notifications/initialized`, and fetches `tools/list`. `mk_tool` builds the callables in `client.tools` from those schemas, reversing the server's `get_schema` conversion.

A bound tool calls `call_text`, which joins text blocks and raises `RuntimeError` for a tool error. Use `call_tool` when you need the complete result dict. JSON-RPC errors raise in `rpc`.

In [ ]:
#| export
class MCPClient:
    "Call an MCP server's tools as Python functions"
    def __init__(self,
        tr, # A transport: `StdioTransport`, `HTTPTransport`, or compatible
    ):
        self.tr,self.tools,self._id = tr,AttrDict(),0
    @classmethod
    @delegates(StdioTransport)
    def stdio(cls, argv, **kwargs): return cls(StdioTransport(argv, **kwargs))
    @classmethod
    @delegates(HTTPTransport)
    def http(cls, url, **kwargs): return cls(HTTPTransport(url, **kwargs))
    async def rpc(self, method, **params):
        "One request round trip, returning its `result`; raises on an error reply"
        self._id += 1
        resp = await self.tr.send(jreq(method, self._id, **params))
        if 'error' in resp: raise RuntimeError(f"{resp['error']['code']}: {resp['error']['message']}")
        return resp['result']
    async def start(self):
        "Handshake, then bind the server's tools; returns `self`"
        await self.tr.start()
        self.info = await self.rpc('initialize', protocolVersion=PROTO_VERSIONS[0], capabilities={},
            clientInfo=dict(name='mcpmini', version=__version__))
        self.tr.proto = self.info['protocolVersion']
        await self.tr.send(jreq('notifications/initialized'))
        for t in (await self.rpc('tools/list'))['tools']: self.tools[t['name']] = mk_tool(self.call_text, dict2obj(t))
        return self
    async def call_tool(self, name, **kw):
        "The raw `tools/call` result dict"
        return await self.rpc('tools/call', name=name, arguments=kw)
    async def call_text(self, name, **kw):
        "The text of a `tools/call` reply; raises if the tool errored"
        res = await self.call_tool(name, **kw)
        txt = '\n'.join(c['text'] for c in res.get('content',[]) if c.get('type')=='text')
        if res.get('isError'): raise RuntimeError(txt)
        return txt
    async def __aenter__(self): return await self.start()
    async def __aexit__(self, *args): await self.tr.aclose()

We can now call the remote converter as a Python function. Its signature and docs come from the server's schema, with parameter descriptions in `Annotated` metadata. Passing it back to `get_schema` recovers the original argument schema. The client also receives the server instructions.

The example calls the converter and uses the async tool's default argument. It then shows how a tool exception and an unknown tool name reach the Python caller:

In [ ]:
async with MCPClient.http(url, token='sesame') as c:
    test_eq(c.info['serverInfo']['name'], 'demo')
    test_eq(c.info['instructions'], 'Convert temperatures on request.')
    test_eq(list(inspect.signature(c.tools.fahrenheit).parameters), ['celsius'])
    test_eq(get_schema(c.tools.fahrenheit, pname='inputSchema')['inputSchema'], srv.specs[0]['inputSchema'])
    test_eq(await c.tools.fahrenheit(celsius=100), '212.0')
    test_eq(await c.tools.nap(), 'napped 1ms')
    with expect_fail(RuntimeError, contains='ZeroDivisionError'): await c.tools.crash()
    with expect_fail(RuntimeError, contains='-32602'): await c.call_tool('flux')
    doc_line = c.tools.fahrenheit.__doc__
doc_line

'Convert Celsius to Fahrenheit\n\nReturns:\n- type: number'

Now launch a stdio server in a subprocess, as an MCP host would. The temporary script imports the exported module and defines its own converter; it cannot use this notebook's in-memory function:

In [ ]:
serverfile = Path(tempfile.mkdtemp())/'demo_stdio.py'
serverfile.write_text('''import asyncio
from mcpmini.core import MCPServer, serve_stdio

def fahrenheit(
    celsius:float, # Temperature to convert
)->float:
    "Convert Celsius to Fahrenheit"
    return celsius*9/5+32

asyncio.run(serve_stdio(MCPServer('demo-stdio', [fahrenheit])))
''')
async with MCPClient.stdio([sys.executable, str(serverfile)]) as c:
    test_eq(c.info['serverInfo']['name'], 'demo-stdio')
    res = await c.tools.fahrenheit(celsius=-40)
res


'-40.0'

## Interop

Talking to ourselves is not enough to establish compatibility. First, the official SDK client initializes our HTTP server, lists its tools, and calls the converter. `streams[:2]` takes the two streams without depending on the optional session-id callback returned by older SDK versions:

In [ ]:
from mcp import ClientSession
from mcp.client.streamable_http import streamable_http_client, create_mcp_http_client

In [ ]:
async with create_mcp_http_client(headers={'Authorization': 'Bearer sesame'}) as hc, \
        streamable_http_client(url, http_client=hc) as streams, \
        ClientSession(*streams[:2]) as s:
    sdk_init = await s.initialize()
    test_eq({t.name for t in (await s.list_tools()).tools}, {'fahrenheit','nap','crash'})
    sdk_res = await s.call_tool('fahrenheit', dict(celsius=100))
test_eq(sdk_res.content[0].text, '212.0')
sdk_init.serverInfo

Implementation(name='demo', title=None, version='0.0.5', websiteUrl=None, icons=None)

In the other direction, our client calls an SDK FastMCP server. This setup returns SSE and creates a session id. The client sends the id with subsequent requests. After calling the tool, it ends the server session explicitly before closing its HTTP connection:

In [ ]:
import logging, uvicorn
from mcp.server.fastmcp import FastMCP


In [ ]:
logging.getLogger('mcp').setLevel(logging.WARNING)
fm = FastMCP('sdkdemo')
@fm.tool()
def double(x: int) -> int:
    "Twice `x`"
    return x*2

port2 = free_port()
sdk_task = asyncio.create_task(uvicorn.Server(uvicorn.Config(fm.streamable_http_app(), host='127.0.0.1', port=port2, log_level='warning')).serve())
for _ in range(100):
    try:
        with socket.create_connection(('127.0.0.1', port2), timeout=0.5): break
    except OSError: await asyncio.sleep(0.05)
async with MCPClient.http(f'http://127.0.0.1:{port2}/mcp') as c:
    test_eq(c.info['serverInfo']['name'], 'sdkdemo')
    assert c.tr.sess, 'SDK server should have minted a session id'
    dbl = await c.tools.double(x=21)
    await c.tr.delete()
    assert c.tr.sess is None
dbl

[09/09/26 18:40:43] INFO     HTTP Request: POST                  _client.py:1740
                             http://127.0.0.1:56411/mcp                         
                             "HTTP/1.1 200 OK"                                  
                    INFO     HTTP Request: POST                  _client.py:1740
                             http://127.0.0.1:56411/mcp                         
                             "HTTP/1.1 202 Accepted"                            
                    INFO     HTTP Request: POST                  _client.py:1740
                             http://127.0.0.1:56411/mcp                         
                             "HTTP/1.1 200 OK"                                  
                    INFO     HTTP Request: POST                  _client.py:1740
                             http://127.0.0.1:56411/mcp                         
                             "HTTP/1.1 200 OK"                                  


'42'

## The mcpmini command

An MCP host needs a command to launch. `mcpmini tools.py` loads and serves a Python file without a wrapper script. The file stem names the server, and its docstring supplies the instructions.

A nonempty `__all__` selects the objects to serve. Without it, `load_server` selects functions defined in that module. Either way, it removes names beginning with `_`. Loading the file executes its top-level code.

In [ ]:
#| export
def _load_mod(path):
    spec = importlib.util.spec_from_file_location(Path(path).stem, path)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

def load_server(
    path, # Python file whose public docmented functions become the tools
):
    "An `MCPServer` for the file at `path`: named after it, instructed by its docstring, serving its functions"
    mod = _load_mod(path)
    names = getattr(mod, '__all__', None)
    fs = [getattr(mod,n) for n in names] if names else [v for v in vars(mod).values() if inspect.isfunction(v) and v.__module__==mod.__name__]
    return MCPServer(mod.__name__, [f for f in fs if not f.__name__.startswith('_')], instructions=mod.__doc__)

This file defines a public converter, a private helper, and an import. With no `__all__`, only the converter becomes a tool. Imported helpers are not tools by default; an explicit `__all__` can include them:

In [ ]:
toolsfile = Path(tempfile.mkdtemp())/'weather.py'
toolsfile.write_text(""""Answer weather questions."
from json import dumps

def _secret(): pass

def celsius(
    fahrenheit:float, # Temperature to convert
)->float:
    "Convert Fahrenheit to Celsius"
    return (fahrenheit-32)*5/9
""")
wsrv = load_server(toolsfile)
test_eq((wsrv.name, list(wsrv.funcs), wsrv.instructions), ('weather', ['celsius'], 'Answer weather questions.'))
wsrv.specs

[{'name': 'celsius',
  'description': 'Convert Fahrenheit to Celsius\n\nReturns:\n- type: number',
  'inputSchema': {'type': 'object',
   'properties': {'fahrenheit': {'description': 'Temperature to convert',
     'type': 'number'}},
   'required': ['fahrenheit']}}]

`call_parse` turns `main`'s docments into the CLI. Stdio is the default. HTTP deployment uses `--transport http`, `--host`, `--port`, and `--path`. Supply the token with `--token` or `$MCPMINI_TOKEN`; `--no-token` opts out of the non-loopback protection. `streamable-http` is another accepted transport spelling.

In [ ]:
#| export
@call_parse
def main(
    tools:str, # Python file whose public docmented functions are served
    transport:str='stdio', # 'stdio', or 'http' for streamable HTTP
    host:str='127.0.0.1', # Interface to bind, for http
    port:int=8000, # Port to bind, for http
    token:str=None, # Bearer token, for http; `$MCPMINI_TOKEN` if unset
    no_token:bool=False, # Serve http on a non-loopback interface openly, when auth lives elsewhere
    path:str='/mcp', # Endpoint path, for http
):
    "Serve a Python file's docmented functions as an MCP server"
    srv = load_server(tools)
    if transport=='stdio': asyncio.run(serve_stdio(srv))
    elif transport in ('http','streamable-http'): asyncio.run(serve_mcp(srv, host=host, port=port, token=token, no_token=no_token, path=path))
    else: raise ValueError(f'unknown transport: {transport}')

This time the client launches the installed `mcpmini` command with the tools file as its argument. We check that the instructions reach the client and that the converter returns the expected value:

In [ ]:
cmd = shutil.which('mcpmini')
assert cmd, 'mcpmini console script not installed: run `uv sync`'
async with MCPClient.stdio([cmd, str(toolsfile)]) as c:
    test_eq(c.info['instructions'], 'Answer weather questions.')
    freezing = await c.tools.celsius(fahrenheit=32)
freezing


'0.0'

## A live client

SDK interoperability does not tell us whether a model host will discover and use the tool. This disabled example uses the Claude Agent SDK to run Claude Code headlessly. Claude launches `mcpmini`, discovers the converter, and calls it.

Checking the answer alone would miss a model doing the conversion itself. The assertion checks the actual `tool_use` name as well. These cells spend model tokens and do not run in automated tests.

In [ ]:
#| eval: false
from claude_agent_sdk import query, ClaudeAgentOptions, AssistantMessage, ResultMessage

In [ ]:
#| eval: false
logging.getLogger('claude_agent_sdk').setLevel(logging.WARNING)
live_tools = Path(tempfile.mkdtemp())/'weather.py'
live_tools.write_text(toolsfile.read_text())
opts = ClaudeAgentOptions(model='haiku', max_turns=3, tools=[], setting_sources=[], strict_mcp_config=True,
    mcp_servers=dict(demo=dict(type='stdio', command=shutil.which('mcpmini'), args=[str(live_tools)])),
    allowed_tools=['mcp__demo__celsius'], system_prompt='Use the tools you are given.')
msgs = [m async for m in query(prompt='Convert 212 fahrenheit to celsius with the celsius tool. Reply with only the number.', options=opts)]
tus = [b.name for m in msgs if isinstance(m, AssistantMessage) for b in m.content if getattr(b,'name',None)]
res = first(m.result for m in msgs if isinstance(m, ResultMessage))
test_eq(tus, ['mcp__demo__celsius'])
assert '100' in res
res

'100.0'

The HTTP version supplies a URL and an `Authorization` header instead of a subprocess command. The server could also run on another machine. To record that connection in Claude Code, `claude mcp add --transport http` accepts the URL and `--header` option. This example still uses loopback and also spends model tokens.

In [ ]:
#| eval: false
live_port = free_port()
live_task = asyncio.create_task(serve_mcp(wsrv, port=live_port, token='sesame'))
await asyncio.sleep(0.2)
hopts = ClaudeAgentOptions(model='haiku', max_turns=3, tools=[], setting_sources=[], strict_mcp_config=True,
    mcp_servers=dict(demo=dict(type='http', url=f'http://127.0.0.1:{live_port}/mcp', headers={'Authorization': 'Bearer sesame'})),
    allowed_tools=['mcp__demo__celsius'], system_prompt='Use the tools you are given.')
hmsgs = [m async for m in query(prompt='Convert 451 fahrenheit to celsius with the celsius tool. Reply with only the number.', options=hopts)]
htus = [b.name for m in hmsgs if isinstance(m, AssistantMessage) for b in m.content if getattr(b,'name',None)]
hres = first(m.result for m in hmsgs if isinstance(m, ResultMessage))
live_task.cancel()
test_eq(htus, ['mcp__demo__celsius'])
assert '232' in hres
hres

'232.78'

## Cleanup

Cancel both loopback servers and wait for their tasks to finish.

In [ ]:
for t in (server_task, sdk_task): t.cancel()
res = await asyncio.gather(server_task, sdk_task, return_exceptions=True)


## Export -

In [ ]:
#|hide
#|eval: false
import nbdev
nbdev.nbdev_export()
